# Cálculo de Métricas de Desproporcionalidade — Projeto VigiMed  
# Disproportionality Metrics Calculation — VigiMed Project

---

## 📘 Descrição | Description

**PT:**  
Este notebook realiza o cálculo das métricas de desproporcionalidade
utilizadas para detecção de possíveis sinais de segurança de medicamentos.

Ele utiliza a base analítica construída no notebook anterior e calcula
as métricas estatísticas empregadas em farmacovigilância para identificação
de associações medicamento–evento.

**EN:**  
This notebook calculates disproportionality metrics used to detect
potential drug safety signals.

It uses the analytical dataset created in the previous notebook and
computes statistical metrics commonly applied in pharmacovigilance to
identify drug–event associations.

---

## 🔄 Fluxo executado | Workflow

**PT — O notebook executa automaticamente:**

1. Upload da base analítica gerada anteriormente
2. Preparação dos pares medicamento–evento
3. Construção das tabelas de contingência
4. Cálculo das métricas estatísticas
5. Aplicação dos critérios de detecção de sinais
6. Exportação dos resultados finais

**EN — The notebook automatically performs:**

1. Upload of the analytical dataset generated previously
2. Preparation of drug–event pairs
3. Construction of contingency tables
4. Calculation of disproportionality metrics
5. Application of signal detection criteria
6. Export of final results

---

## ▶️ Instruções de uso | Usage instructions

**PT:**

1. Execute primeiro o notebook de construção do banco.
2. Baixe o arquivo `fat_analitica.parquet` gerado.
3. Faça upload do arquivo quando solicitado neste notebook.
4. Execute todas as células.

**EN:**

1. Run the database construction notebook first.
2. Download the generated `fat_analitica.parquet` file.
3. Upload the file when requested in this notebook.
4. Run all notebook cells.

---
## 📥 Entrada necessária | Required input

**PT:**  
Arquivo gerado pelo notebook anterior:

**EN:**  
File generated by the previous notebook:


fat_analitica.parquet


---


## 📤 Resultado gerado | Output generated

**PT:**  
Arquivo contendo as métricas calculadas e os sinais detectados:

**EN:**  
File containing calculated metrics and detected safety signals:


sinais_detectados.parquet



---

## 📧 Contato | Contact

Ana Carolina Jacoby  
Email: anacarolinajacoby0@gmail.com



## 1- Instalação das dependências | Installing dependencies

**PT:**  
Esta etapa instala automaticamente os programas necessários para executar
o notebook.

**EN:**  
This step automatically installs the required software packages needed to
run the notebook.


In [ ]:
print("Instalando dependências...")

!pip install duckdb pandas pyarrow --quiet

print("Dependências instaladas com sucesso.")


## 2- Importação das bibliotecas | Importing libraries

**PT:**  
Carrega as bibliotecas utilizadas para manipulação dos dados e cálculo das
métricas estatísticas.

**EN:**  
Loads the libraries used for data manipulation and statistical calculations.


In [ ]:
import duckdb
import pandas as pd
from pathlib import Path
import os

print("Bibliotecas carregadas com sucesso.")


## 2.1- Registro do ambiente de execução | Execution environment record

**PT:**  
Esta etapa registra as versões das bibliotecas utilizadas para permitir a
reprodução futura dos resultados.

**EN:**  
This step records the versions of libraries used to allow future
reproducibility of the results.


In [ ]:
import sys
import duckdb
import pandas as pd
import numpy as np

print("Versões do ambiente:")
print("Python:", sys.version)
print("duckdb:", duckdb.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


## 3- Upload da base analítica | Upload analytical dataset

**PT:**  
Faça upload do arquivo `fat_analitica.parquet`, gerado no notebook de
construção do banco.

Este arquivo será utilizado para cálculo das métricas.

**EN:**  
Upload the file `fat_analitica.parquet`, generated in the database
construction notebook.

This file will be used to compute disproportionality metrics.


In [ ]:
from google.colab import files

print("Selecione o arquivo fat_analitica.parquet para upload:")

uploaded = files.upload()

print("Upload concluído.")


## 4- Verificação do arquivo enviado | Uploaded file check

**PT:**  
Esta etapa verifica se o arquivo necessário foi enviado corretamente.

Caso o arquivo esteja ausente ou com nome diferente, a execução será
interrompida para evitar erros posteriores.

**EN:**  
This step checks whether the required file was correctly uploaded.

If the file is missing or has a different name, execution stops to avoid
later errors.


In [ ]:
required_file = "fat_analitica.parquet"

if required_file not in uploaded:
    raise ValueError(f"""
Arquivo necessário não encontrado: {required_file}

Por favor, faça upload do arquivo correto e execute novamente.

Required file not found: {required_file}
Please upload the correct file and run again.
""")

print("Arquivo correto enviado.")


## 5- Carregamento dos dados | Loading dataset

**PT:**  
Nesta etapa o arquivo enviado é carregado no banco de dados analítico,
permitindo a execução das análises estatísticas.

**EN:**  
In this step, the uploaded file is loaded into the analytical database,
allowing statistical analyses to be executed.


In [ ]:
import duckdb

print("Inicializando banco de dados...")

con = duckdb.connect("vigimed_calculos.duckdb")

print("Carregando base analítica...")

con.execute("""
CREATE OR REPLACE TABLE fat_analitica AS
SELECT * FROM read_parquet('fat_analitica.parquet')
""")

print("Dados carregados com sucesso.")


## 6- Construção das tabelas de contingência | Contingency table construction

**PT:**  
Nesta etapa são calculadas as contagens necessárias para construir as
tabelas de contingência utilizadas nos cálculos de desproporcionalidade.

Para cada par medicamento–evento são obtidas as contagens:

a: medicamento + evento  
b: medicamento + outros eventos  
c: outros medicamentos + evento  
d: outros medicamentos + outros eventos  

**EN:**  
In this step, counts required to build contingency tables used in
disproportionality calculations are computed.

For each drug–event pair, the following counts are obtained:

a: drug + event  
b: drug + other events  
c: other drugs + event  
d: other drugs + other events




## 6.1 - Carregamento em pandas para facilitar a construção

In [ ]:
import pandas as pd
import numpy as np

print("Carregando dados para construção da matriz...")

df = con.execute("""
SELECT DISTINCT
    ID,
    ATC_LEVEL_5,
    REACAO_PT
FROM fat_analitica
WHERE ATC_LEVEL_5 IS NOT NULL
  AND REACAO_PT IS NOT NULL
""").fetchdf()

print("Dados carregados.")


## 6.2 Construção da matriz

In [ ]:
def build_matrix(df):

    x = df.drop_duplicates(["ID", "ATC_LEVEL_5", "REACAO_PT"])

    n_total = x["ID"].nunique()

    n_drug = x.groupby("ATC_LEVEL_5")["ID"].nunique()
    n_event = x.groupby("REACAO_PT")["ID"].nunique()

    pairs = x.groupby(
        ["ATC_LEVEL_5", "REACAO_PT"]
    )["ID"].nunique()

    df_out = pairs.reset_index(name="a")

    df_out["b"] = df_out["ATC_LEVEL_5"].map(n_drug) - df_out["a"]
    df_out["c"] = df_out["REACAO_PT"].map(n_event) - df_out["a"]

    df_out["d"] = (
        n_total
        - df_out["a"]
        - df_out["b"]
        - df_out["c"]
    )

    df_out["n_total"] = n_total

    return df_out

print("Construindo matriz de contingência...")

df_matrix = build_matrix(df)

print("Matriz construída.")

# Verificação de segurança
if df_matrix.empty:
    raise ValueError(
        "Nenhum dado disponível após preparação. "
        "Verifique se o arquivo enviado contém registros válidos."
    )

print(f"Número de pares medicamento–evento: {len(df_matrix):,}")
print(f"Número total de notificações: {df_matrix.n_total.iloc[0]:,}")


## 7- Cálculo da Razão de Chances de Notificação (ROR) | Reporting Odds Ratio Calculation

**PT:**  
A Razão de Chances de Notificação (Reporting Odds Ratio – ROR) é uma das
métricas mais utilizadas em farmacovigilância para detectar sinais de
notificação desproporcional de reações adversas a medicamentos.

Ela compara a chance de um evento adverso ser notificado para um
determinado medicamento com a chance de notificação do mesmo evento para
todos os demais medicamentos na base de dados.

Essa métrica auxilia na identificação de potenciais problemas de segurança,
mas **não estabelece causalidade**, apenas sinaliza associações que merecem
investigação adicional.

### Interpretação

- **ROR = 1**: Não há desproporcionalidade na notificação.
- **ROR > 1**: Indica maior frequência relativa de notificação,
  sugerindo possível sinal de segurança.
- **ROR < 1**: Indica menor frequência relativa de notificação.

### Intervalo de confiança (95%)

Para que um ROR seja considerado estatisticamente significativo,
o **limite inferior do intervalo de confiança de 95% deve ser maior que 1**,
indicando que a associação observada provavelmente não ocorreu ao acaso.

---

**EN:**  
The Reporting Odds Ratio (ROR) is one of the most commonly used metrics in
pharmacovigilance to detect signals of disproportionate reporting of adverse
drug reactions.

It compares the odds of reporting a specific adverse event for a given drug
with the odds of reporting the same event for all other drugs in the
database.

This metric helps identify potential safety concerns but **does not imply
causality**, only statistical association requiring further investigation.

### Interpretation

- **ROR = 1**: No disproportionality in reporting.
- **ROR > 1**: Indicates increased reporting frequency, suggesting a
  potential safety signal.
- **ROR < 1**: Indicates lower reporting frequency.

### 95% Confidence Interval

An ROR is considered statistically significant when the **lower bound of
the 95% confidence interval is greater than 1**, suggesting the association
is unlikely due to chance.


In [ ]:
def calculate_ror(a, b, c, d):
    """
    Calculate Reporting Odds Ratio (ROR) and its 95% confidence interval.
    """

    # Correção de Haldane-Anscombe
    a_c, b_c, c_c, d_c = a + 0.5, b + 0.5, c + 0.5, d + 0.5

    ror = (a_c * d_c) / (b_c * c_c)

    se_log = np.sqrt(
        1/a_c + 1/b_c + 1/c_c + 1/d_c
    )

    ci_low = np.exp(np.log(ror) - 1.96 * se_log)
    ci_high = np.exp(np.log(ror) + 1.96 * se_log)

    return ror, ci_low, ci_high


### 7.1- Aplicação do cálculo do ROR | Applying ROR calculation

**PT:**  
Nesta etapa aplicamos a função de cálculo do ROR a cada par
medicamento–evento presente na matriz de contingência.

O resultado é adicionado como novas colunas contendo o valor do ROR e seu
intervalo de confiança.

**EN:**  
In this step, the ROR calculation function is applied to each drug–event
pair in the contingency matrix.

The resulting ROR values and confidence intervals are added as new columns.


In [ ]:
print("Calculando ROR para todos os pares...")

def apply_ror(row):
    ror, ci_low, ci_high = calculate_ror(
        row.a, row.b, row.c, row.d
    )

    return pd.Series({
        "ROR": ror,
        "ROR_Lower_CI": ci_low,
        "ROR_Upper_CI": ci_high
    })

ror_results = df_matrix.apply(apply_ror, axis=1)

df_ror = pd.concat([df_matrix, ror_results], axis=1)

print("ROR calculado com sucesso.")


## 8- Cálculo do PRR | PRR Calculation

**PT:**  
Nesta etapa é definida a função para cálculo do Proportional Reporting Ratio
(PRR) e seu intervalo de confiança para pares medicamento–evento.

O PRR mede se um evento adverso é relatado proporcionalmente com maior
frequência para um medicamento em comparação com todos os demais
medicamentos na base de dados.

Esta métrica indica **desproporcionalidade de notificação**, mas não
estabelece causalidade.

### Interpretação

- **PRR = 1**: ausência de associação.
- **PRR > 1**: maior frequência relativa do evento para o medicamento.
- **PRR < 1**: menor frequência relativa.

### Intervalo de confiança (95%)

O PRR é considerado estatisticamente significativo quando o limite inferior
do intervalo de confiança é maior ou igual a 1.

Critérios adicionais de sinal (ex.: número mínimo de casos e valor de
Qui-quadrado) são aplicados posteriormente.

---

**EN:**  
This step defines the function used to calculate the Proportional Reporting
Ratio (PRR) and its confidence interval for drug–event pairs.

PRR measures whether an adverse event is reported proportionally more often
for a drug compared to all other drugs in the database.

It indicates reporting disproportionality but does not establish causality.

### Interpretation

- **PRR = 1**: no association.
- **PRR > 1**: increased reporting frequency.
- **PRR < 1**: decreased reporting frequency.

### 95% Confidence Interval

PRR is considered statistically significant when the lower bound of the
confidence interval is greater than or equal to 1.

Additional signal criteria are applied later.


In [ ]:
def calculate_prr(a, b, c, d):
    """
    Calculate PRR, confidence interval and chi-square statistic.
    """

    if (a + b == 0) or (c + d == 0):
        return np.nan, np.nan, np.nan, np.nan

    p1 = a / (a + b)
    p2 = c / (c + d)

    if p2 == 0:
        return np.nan, np.nan, np.nan, np.nan

    prr = p1 / p2

    # correção apenas para erro padrão
    a_c, b_c, c_c, d_c = a+0.5, b+0.5, c+0.5, d+0.5

    se_log = np.sqrt(
        (1/a_c) - (1/(a_c+b_c)) +
        (1/c_c) - (1/(c_c+d_c))
    )

    ci_low = np.exp(np.log(prr) - 1.96 * se_log)
    ci_high = np.exp(np.log(prr) + 1.96 * se_log)

    # Qui-quadrado
    N = a + b + c + d
    denom = (a+b)*(c+d)*(a+c)*(b+d)

    chi2 = np.nan if denom == 0 else \
        (N * (a*d - b*c)**2) / denom

    return prr, ci_low, ci_high, chi2


### 8.1- Aplicação do cálculo do PRR | Applying PRR calculation

**PT:**  
Nesta etapa aplicamos a função de cálculo do PRR a cada par
medicamento–evento presente na matriz de contingência.

O resultado inclui o valor do PRR, seu intervalo de confiança e o valor
do teste Qui-quadrado.

**EN:**  
In this step, the PRR calculation function is applied to each drug–event
pair in the contingency matrix.

The result includes PRR values, confidence intervals and the Chi-square
statistic.


In [ ]:
print("Calculando PRR para todos os pares...")

def apply_prr(row):
    prr, ci_low, ci_high, chi2 = calculate_prr(
        row.a, row.b, row.c, row.d
    )

    return pd.Series({
        "PRR": prr,
        "PRR_Lower_CI": ci_low,
        "PRR_Upper_CI": ci_high,
        "PRR_Chi2": chi2
    })

prr_results = df_matrix.apply(apply_prr, axis=1)

df_prr = pd.concat([df_matrix, prr_results], axis=1)

print("PRR calculado com sucesso.")


## 9-  Medidas Bayesianas de Desproporcionalidade | Bayesian Disproportionality Measures

**PT:**  
Nesta etapa são implementadas métricas baseadas em inferência Bayesiana,
em particular o Componente de Informação (IC), utilizado pelo algoritmo
BCPNN da Organização Mundial da Saúde.

Diferentemente das métricas frequentistas, abordagens Bayesianas aplicam
mecanismos de "encolhimento" (shrinkage), tornando estimativas mais
estáveis quando o número de notificações é pequeno e reduzindo sinais
falsos positivos.

---

**EN:**  
In this step, Bayesian disproportionality measures are implemented,
specifically the Information Component (IC) used by the WHO BCPNN
algorithm.

Unlike frequentist metrics, Bayesian approaches apply shrinkage mechanisms,
increasing stability in small-count situations and reducing false positive
signals.


### 9.1 Componente de Informação (IC) | Information Component (IC)

**PT:**  
O IC compara a probabilidade observada de um par medicamento–evento com a
probabilidade esperada na base de dados, utilizando logaritmo base 2.

Valores positivos indicam que o evento ocorre mais frequentemente do que o
esperado.

Um possível sinal é considerado quando o limite inferior do intervalo de
credibilidade é maior que zero.

---

**EN:**  
The IC compares the observed probability of a drug–event pair with the
expected probability in the database using base-2 logarithm.

Positive values indicate higher-than-expected reporting frequency.

A potential signal occurs when the lower bound of the credibility interval
is greater than zero.


In [ ]:
def calculate_ic(a, b, c, d):
    """
    Calculate Information Component (IC) and credibility interval.
    """

    n = a + b + c + d

    if n <= 0:
        return np.nan, np.nan, np.nan

    expected = ((a + b) * (a + c)) / n

    if expected <= 0:
        return np.nan, np.nan, np.nan

    # IC com correção bayesiana
    ic = np.log2((a + 0.5) / (expected + 0.5))

    numerador = max(n - a, 0)
    denominador = (a + 0.5) * (n + 1)

    se_ic = (1 / np.log(2)) * np.sqrt(
        numerador / denominador
    )

    ic_low = ic - 1.96 * se_ic
    ic_high = ic + 1.96 * se_ic

    return ic, ic_low, ic_high


### 9.1.1- Aplicação do cálculo do IC | Applying IC calculation

**PT:**  
Nesta etapa aplicamos o cálculo do Componente de Informação (IC) a cada
par medicamento–evento presente na matriz de contingência.

O resultado inclui o valor do IC e seus limites inferior e superior do
intervalo de credibilidade.

**EN:**  
In this step, the Information Component (IC) calculation is applied to each
drug–event pair in the contingency matrix.

The result includes IC values and the lower and upper bounds of the
credibility interval.


In [ ]:
print("Calculando IC para todos os pares...")

def apply_ic(row):
    ic, ic_low, ic_high = calculate_ic(
        row.a, row.b, row.c, row.d
    )

    return pd.Series({
        "IC_BCPNN": ic,
        "IC_Lower_CI": ic_low,
        "IC_Upper_CI": ic_high
    })

ic_results = df_matrix.apply(apply_ic, axis=1)

df_ic = pd.concat([df_matrix, ic_results], axis=1)

print("IC calculado com sucesso.")


### 10- Média Geométrica Bayesiana Empírica (EBGM) | Empirical Bayes Geometric Mean (EBGM)

**PT:**  
O EBGM é uma medida baseada no modelo Gamma-Poisson utilizada para ajustar
estimativas de desproporcionalidade, reduzindo a instabilidade causada por
pequenos números de notificações.

O valor EBGM representa uma estimativa estabilizada da razão entre o número
de relatos observados e esperados.

### Interpretação

- **EBGM ≈ 1**: ocorrência compatível com o esperado.
- **EBGM > 1**: evento relatado com maior frequência que o esperado.

O limite inferior do intervalo de credibilidade de 90%, denominado EB05,
é frequentemente utilizado como critério de sinal.

---

**EN:**  
EBGM is based on the Gamma-Poisson model and provides a stabilized estimate
of disproportionality, reducing variability caused by small counts.

EBGM represents a shrinkage-adjusted ratio between observed and expected
reports.

### Interpretation

- **EBGM ≈ 1**: reporting frequency consistent with expectation.
- **EBGM > 1**: higher-than-expected reporting.

The lower bound of the 90% credibility interval, called EB05, is commonly
used for signal detection.


In [ ]:
def calculate_ebgm(a, b, c, d):
    """
    Calculate EBGM and 90% credibility interval.
    """

    n = a + b + c + d

    if n <= 0:
        return np.nan, np.nan, np.nan

    expected = ((a + b) * (a + c)) / n

    if expected <= 0:
        return np.nan, np.nan, np.nan

    ebgm = (a + 0.5) / (expected + 0.5)

    se = np.sqrt(
        1/(a + 0.5) +
        1/(expected + 0.5)
    )

    eb05 = ebgm * np.exp(-1.645 * se)
    eb95 = ebgm * np.exp(1.645 * se)

    return ebgm, eb05, eb95


### 10.1- Aplicação do cálculo do EBGM | Applying EBGM calculation

**PT:**  
Nesta etapa aplicamos o cálculo do EBGM a cada par medicamento–evento da
matriz de contingência.

**EN:**  
In this step, EBGM is calculated for each drug–event pair in the
contingency matrix.


In [ ]:
print("Calculando EBGM para todos os pares...")

def apply_ebgm(row):
    ebgm, e05, e95 = calculate_ebgm(
        row.a, row.b, row.c, row.d
    )

    return pd.Series({
        "EBGM_MGPS": ebgm,
        "EBGM_E05": e05,
        "EBGM_E95": e95
    })

ebgm_results = df_matrix.apply(apply_ebgm, axis=1)

df_ebgm = pd.concat([df_matrix, ebgm_results], axis=1)

print("EBGM calculado com sucesso.")


## 11- Consolidação das métricas calculadas | Metrics consolidation

**PT:**  
Nesta etapa final, reunimos todas as métricas calculadas em uma única
tabela, contendo as contagens da matriz de contingência e as métricas de
desproporcionalidade frequentistas e bayesianas.

Esta tabela será utilizada para exportação e análises posteriores.

**EN:**  
In this final step, all calculated metrics are consolidated into a single
table containing contingency counts and both frequentist and Bayesian
disproportionality measures.

This table will be used for export and subsequent analyses.



In [ ]:
print("Consolidando métricas...")

df_final = (
    df_matrix
    .merge(df_ror.drop(columns=["a","b","c","d","n_total"], errors="ignore"),
           on=["ATC_LEVEL_5","REACAO_PT"], how="left")
    .merge(df_prr.drop(columns=["a","b","c","d","n_total"], errors="ignore"),
           on=["ATC_LEVEL_5","REACAO_PT"], how="left")
    .merge(df_ic.drop(columns=["a","b","c","d","n_total"], errors="ignore"),
           on=["ATC_LEVEL_5","REACAO_PT"], how="left")
    .merge(df_ebgm.drop(columns=["a","b","c","d","n_total"], errors="ignore"),
           on=["ATC_LEVEL_5","REACAO_PT"], how="left")
)

print("Métricas consolidadas com sucesso.")


In [ ]:
print(f"Total de registros na tabela final: {len(df_final):,}")


## 12- Exportação e download dos resultados | Export and download results

**PT:**  
O dataset completo é salvo e disponibilizado para download.  
Após a execução, o navegador iniciará automaticamente o download do arquivo.

**EN:**  
The full dataset is saved and made available for download.  
After execution, your browser will automatically download the file.


In [ ]:
from google.colab import files

print("Salvando resultados...")

# Exportação em Parquet (uso analítico)
df_final.to_parquet("metricas_completas.parquet", index=False)

# Exportação em CSV (uso em Excel e leitura simples)
df_final.to_csv("metricas_completas.csv", index=False)

print("Arquivos gerados com sucesso.")
print("Iniciando downloads...")

files.download("metricas_completas.parquet")
files.download("metricas_completas.csv")



## Processo concluído | Process completed


In [ ]:
print("""
Processo concluído com sucesso!

Todas as métricas foram calculadas e o dataset final foi exportado.

Agora você pode aplicar filtros específicos ou seguir para etapas
de análise e visualização.
""")
